# exp-023: V2 — 임베딩 + 학습 head (Qwen3-0.6B vs KURE-v1)

- **목적:** 처음으로 *진짜 AI 모델 학습* 단계 — Qwen3-Embedding-0.6B / KURE-v1 임베딩 위에 Linear/MLP head를 학습해 5관점 매칭 점수 생성. 5컴포넌트 rule baseline(exp-018) 대비 의미 신호가 ranking을 어떻게 바꾸는지 검증.
- **차별성 축:** ① 자소서 정성 (임베딩으로 STAR 의미 신호 직접 사용), ② 한국어 (Qwen3/KURE 두 한국어 백본 비교), ⑤ 매칭 head 학습 (LLM API 0건)
- **입력 데이터:** `raw/data/user_data.csv` (999명) + `raw/data/company_jobdescription_enriched.partial.csv` (3,000 JD)
- **출력 위치:** `raw/experiments/exp-023-v2-embedding-fusion/`
- **관련 위키:** exp-023-v2-embedding-fusion-head (분석), exp-018-learnable-fusion-head (V1 비교), exp-022-saturation-deep-analysis (왜 V2가 필요한가)
- **작성일:** 2026-06-05
- **시드:** 42 (전 단계 동일)
- **LLM 호출:** ❌ 0건 (임베딩 모델은 inference만)

---

## 재현성 원칙

| 항목 | 설정 |
|---|---|
| seed | 42 (numpy / torch / python random / PYTHONHASHSEED) |
| 임베딩 모델 | Qwen3-Embedding-0.6B, KURE-v1 (HF 캐시 사용) |
| 풀링 | last_token (Qwen3 default), mean (KURE) |
| device | MPS (Apple Silicon GPU) |
| 데이터 분할 | userId 기준 train 800 / held-out 199 (exp-018과 동일) |
| 라벨 | 순환 라벨 학습 + 독립 라벨(exp-019/020) 별도 평가 |
| 임베딩 캐시 | npz로 저장 (`raw/experiments/exp-023-v2-embedding-fusion/embeddings/`) |

## Sweep 설계 (최고 성능 탐색)

| 차원 | 옵션 |
|---|---|
| 백본 | Qwen3-0.6B / KURE-v1 |
| 아키텍처 | Linear / MLP-128 |
| 손실 함수 | MSE / MarginRanking |
| L2 | 0 / 1e-4 / 1e-3 |
| head 구조 | 통합(관점 E) / 5관점 분기 |

→ 2 × 2 × 2 × 3 × 2 = 48 configurations. 시간 제약 시 핵심 차원만 sweep.

In [ ]:
# ---- 환경 + 시드 (재현성 1순위) ----
import os, sys, random, json, re, ast, time, hashlib
import numpy as np
import pandas as pd
from pathlib import Path
import torch

SEED = 42
os.environ['PYTHONHASHSEED'] = str(SEED)
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)
if torch.backends.mps.is_available(): torch.mps.manual_seed(SEED)
DEVICE = torch.device('mps' if torch.backends.mps.is_available() else 'cpu')

ROOT = Path('.')
RAW = ROOT / 'raw' / 'data'
EXP = ROOT / 'raw' / 'experiments'
OUT_DIR = EXP / 'exp-023-v2-embedding-fusion'
EMB_DIR = OUT_DIR / 'embeddings'
OUT_DIR.mkdir(parents=True, exist_ok=True)
EMB_DIR.mkdir(parents=True, exist_ok=True)

import transformers, sentence_transformers
ENV = {
    'python': sys.version.split()[0],
    'torch': torch.__version__,
    'transformers': transformers.__version__,
    'sentence_transformers': sentence_transformers.__version__,
    'device': str(DEVICE),
    'seed': SEED,
    'date': '2026-06-05',
}
print('ENV:', json.dumps(ENV, indent=2))
print('OUT_DIR:', OUT_DIR.relative_to(ROOT))

In [ ]:
# ---- 데이터 로드 (exp-018/021/022와 동일) ----
def parse_list(v):
    if v is None or (isinstance(v, float) and pd.isna(v)): return []
    if isinstance(v, list): return v
    s = str(v).strip()
    if not s or s == '[]': return []
    try:
        x = json.loads(s); return x if isinstance(x, list) else [str(x)]
    except Exception:
        try:
            x = ast.literal_eval(s); return x if isinstance(x, list) else [str(x)]
        except Exception: return [s]
def norm(s): return str(s).strip().lower() if s is not None and not (isinstance(s, float) and pd.isna(s)) else ''
def norm_ind(s):
    s = norm(s); return s[:-1] if s.endswith('s') else s
KO = re.compile(r'[가-힣A-Za-z]{2,}')

ud = pd.read_csv(RAW / 'user_data.csv')
jd = pd.read_csv(RAW / 'company_jobdescription_enriched.partial.csv')
print(f'user rows: {len(ud):,}  unique users: {ud["userId"].nunique():,}')
print(f'jd rows: {len(jd):,}')

In [ ]:
# ---- User / JD 텍스트 구성 (임베딩 입력) ----
def build_user_text(g):
    """한 user의 모든 자소서 행 → 단일 텍스트."""
    r0 = g.iloc[0]
    parts = []
    jobs = [norm(r0[c]) for c in ['interestedJobs_1','interestedJobs_2','interestedJobs_3'] if norm(r0[c])]
    inds = [norm(r0[c]) for c in ['interestedIndustries_1','interestedIndustries_2','interestedIndustries_3'] if norm(r0[c])]
    if jobs: parts.append('관심 직무: ' + ', '.join(jobs))
    if inds: parts.append('관심 산업: ' + ', '.join(inds))
    akws, skills = set(), []
    for _, row in g.iterrows():
        for i in range(3):
            kw = norm(row.get(f'ability_{i}_keyword'))
            if kw: akws.add(kw)
            nm = row.get(f'ability_{i}_name')
            if isinstance(nm, str) and nm.strip(): skills.append(nm.strip())
    if akws: parts.append('보유 역량: ' + ', '.join(sorted(akws)))
    if skills: parts.append('역량 상세: ' + ', '.join(skills[:10]))
    # STAR — 최대 3개 자소서, 각 400자
    stars = []
    for _, row in g.head(3).iterrows():
        seg = []
        for c in ['Situation','Task','Action','Reason','Result']:
            v = row.get(c)
            if isinstance(v, str) and v.strip(): seg.append(v.strip())
        if seg: stars.append(' '.join(seg)[:400])
    if stars: parts.append('자기소개서: ' + ' / '.join(stars))
    return '\n'.join(parts)

def build_jd_text(r):
    parts = []
    if r.get('jd_job_role') and r['jd_job_role'] != 'unknown': parts.append(f'직무: {r["jd_job_role"]}')
    if r.get('jd_industry') and r['jd_industry'] != 'unknown': parts.append(f'산업: {r["jd_industry"]}')
    for col, lbl, lim in [('jd_summary','요약',300), ('jd_main_duties_text','주요업무',500),
                          ('jd_ideal_candidate_text','인재상',300), ('jd_role_purpose','역할목적',200)]:
        v = r.get(col)
        if isinstance(v, str) and v.strip(): parts.append(f'{lbl}: {v.strip()[:lim]}')
    skills = parse_list(r.get('jd_required_skills'))
    if skills: parts.append('필수 스킬: ' + ', '.join(map(str, skills[:10])))
    comp = parse_list(r.get('jd_competencies'))
    if comp: parts.append('필요 역량: ' + ', '.join(map(str, comp[:8])))
    if isinstance(r.get('title'), str): parts.insert(0, f'공고제목: {r["title"]}')
    return '\n'.join(parts)

all_uids = sorted(ud['userId'].unique())
user_texts = {uid: build_user_text(g) for uid, g in ud.groupby('userId')}
jd_texts   = {int(r['job_id']): build_jd_text(r) for _, r in jd.iterrows()}
all_jids   = np.array(sorted(jd_texts.keys()))
N_USERS, N_JDS = len(all_uids), len(all_jids)

# 캐시 무결성 hash (텍스트 내용 변하면 임베딩 재계산)
user_text_hash = hashlib.md5(json.dumps([user_texts[u] for u in all_uids[:20]], ensure_ascii=False).encode()).hexdigest()[:8]
jd_text_hash   = hashlib.md5(json.dumps([jd_texts[int(j)] for j in all_jids[:20]], ensure_ascii=False).encode()).hexdigest()[:8]
print(f'users: {N_USERS}, jds: {N_JDS}')
print(f'user text 평균 길이: {np.mean([len(t) for t in user_texts.values()]):.0f} chars')
print(f'jd text 평균 길이: {np.mean([len(t) for t in jd_texts.values()]):.0f} chars')
print(f'cache hash — user: {user_text_hash}, jd: {jd_text_hash}')

In [ ]:
# ---- 5컴포넌트 + 라벨 행렬 (exp-018과 완전 동일) ----
def build_user_profile(g):
    r0 = g.iloc[0]
    jobs = [norm(r0[c]) for c in ['interestedJobs_1','interestedJobs_2','interestedJobs_3'] if norm(r0[c])]
    inds = [norm_ind(r0[c]) for c in ['interestedIndustries_1','interestedIndustries_2','interestedIndustries_3'] if norm(r0[c])]
    akw, star, skill = set(), [], []
    for _, row in g.iterrows():
        for i in range(3):
            kw = norm(row.get(f'ability_{i}_keyword'))
            if kw: akw.add(kw)
            nm = row.get(f'ability_{i}_name')
            if isinstance(nm, str) and nm.strip(): skill.append(nm)
        for c in ['Situation','Task','Action','Reason','Result']:
            v = row.get(c)
            if isinstance(v, str) and v.strip(): star.append(v)
    st = ' '.join(star)
    return dict(jobs=jobs, industries=inds, ability_kw=akw,
                star_tokens=set(t.lower() for t in KO.findall(st)),
                skill_tokens=set(t.lower() for t in KO.findall(' '.join(skill) + ' ' + st)))

def build_jd_profile(r):
    req  = [norm(x) for x in parse_list(r.get('jd_required_skills')) + parse_list(r.get('jd_preferred_skills')) if norm(x)]
    comp = [norm(x) for x in parse_list(r.get('jd_competencies')) if norm(x)]
    summ = ' '.join([str(r.get(c)) for c in ['jd_summary','jd_main_duties_text','jd_ideal_candidate_text'] if isinstance(r.get(c), str)])
    req_tokens = set()
    for s in req: req_tokens |= set(s.split())
    return dict(role=norm(r.get('jd_job_role')), role2=norm(r.get('jd_job_role_secondary')),
                industry=norm_ind(r.get('jd_industry')), req=set(req), req_tokens=req_tokens,
                comp=set(comp), summary_tokens=set(t.lower() for t in KO.findall(summ)))

UP = {uid: build_user_profile(g) for uid, g in ud.groupby('userId')}
JP = {int(r['job_id']): build_jd_profile(r) for _, r in jd.iterrows()}

PERSPS = list('ABCDE')
COMP_COLS = ['role_match','hard_skill','industry_match','star_overlap','competency']

def components(u, j):
    s_role  = 1.0 if (j['role'] and j['role'] not in ('','unknown') and j['role'] in u['jobs']) \
              else (0.5 if (j['role2'] and j['role2'] not in ('','unknown') and j['role2'] in u['jobs']) else 0.0)
    s_ind   = 1.0 if (j['industry'] and j['industry'] not in ('','unknown') and j['industry'] in u['industries']) else 0.0
    s_skill = min(len(j['req_tokens'] & u['skill_tokens']) / max(len(j['req_tokens']), 1), 1.0) if j['req_tokens'] else 0.0
    s_star  = min(len(u['star_tokens'] & j['summary_tokens']) / max(len(j['summary_tokens']), 1) * 3, 1.0) \
              if (j['summary_tokens'] and u['star_tokens']) else 0.0
    s_comp  = len(j['comp'] & u['ability_kw']) / max(len(j['comp']), 1) if (j['comp'] and u['ability_kw']) else 0.0
    return np.array([s_role, s_skill, s_ind, s_star, s_comp], dtype=np.float32)

PERSP_W = {
    'A': dict(role=.55, ind=.15, skill=.15, comp=.15, star=.00),
    'B': dict(role=.15, ind=.00, skill=.20, comp=.25, star=.40),
    'C': dict(role=.20, ind=.00, skill=.35, comp=.35, star=.10),
    'D': dict(role=.20, ind=.45, skill=.00, comp=.15, star=.20),
    'E': dict(role=.20, ind=.20, skill=.20, comp=.20, star=.20),
}
THRESH = [0.15, 0.32, 0.52, 0.75]

def label_matrix_for_persp(C, p):
    w = PERSP_W[p]
    r = (w['role']*C[:,:,0] + w['skill']*C[:,:,1] + w['ind']*C[:,:,2] + w['star']*C[:,:,3] + w['comp']*C[:,:,4])
    r[(C[:,:,0]==0) & (C[:,:,2]==0)] *= 0.5
    r[(C[:,:,0]==1) & (C[:,:,2]==1)] = np.minimum(1.0, r[(C[:,:,0]==1) & (C[:,:,2]==1)] + 0.10)
    r[(C[:,:,0]==1) & (C[:,:,1]==0) & (C[:,:,4]==0)] *= 0.8
    L = np.zeros_like(r, dtype=np.int8)
    for t in THRESH: L += (r >= t).astype(np.int8)
    return L

print('컴포넌트 행렬 계산 중 (999 × 3000)...')
t0 = time.time()
COMP_MAT = np.zeros((N_USERS, N_JDS, 5), dtype=np.float32)
for i, uid in enumerate(all_uids):
    u = UP[uid]
    for k, jid in enumerate(all_jids):
        COMP_MAT[i, k] = components(u, JP[int(jid)])
LABEL_MAT = {p: label_matrix_for_persp(COMP_MAT.copy(), p) for p in PERSPS}
print(f'완료: {time.time()-t0:.1f}s  COMP_MAT={COMP_MAT.shape}')

---
## 임베딩 사전계산 (캐시 우선)

In [ ]:
from transformers import AutoTokenizer, AutoModel
from torch.utils.data import DataLoader

def last_token_pool(last_hidden_states, attention_mask):
    """Qwen3-Embedding 공식 풀링: 마지막 non-pad token."""
    left_padding = (attention_mask[:, -1].sum() == attention_mask.shape[0])
    if left_padding:
        return last_hidden_states[:, -1]
    seq_lens = attention_mask.sum(dim=1) - 1
    batch = last_hidden_states.shape[0]
    return last_hidden_states[torch.arange(batch, device=last_hidden_states.device), seq_lens]

def mean_pool(last_hidden_states, attention_mask):
    mask = attention_mask.unsqueeze(-1).float()
    summed = (last_hidden_states * mask).sum(dim=1)
    counts = mask.sum(dim=1).clamp(min=1e-9)
    return summed / counts

def encode_texts(texts, model_name, pool='last', max_len=512, batch_size=8):
    """임베딩 batch 인코딩 (MPS)."""
    tok = AutoTokenizer.from_pretrained(model_name, padding_side='left' if pool == 'last' else 'right')
    if tok.pad_token is None: tok.pad_token = tok.eos_token
    mdl = AutoModel.from_pretrained(model_name, torch_dtype=torch.float32).to(DEVICE)
    mdl.eval()
    embs = []
    pool_fn = last_token_pool if pool == 'last' else mean_pool
    with torch.no_grad():
        for i in range(0, len(texts), batch_size):
            batch = texts[i:i+batch_size]
            enc = tok(batch, padding=True, truncation=True, max_length=max_len, return_tensors='pt').to(DEVICE)
            out = mdl(**enc)
            emb = pool_fn(out.last_hidden_state, enc.attention_mask)
            emb = torch.nn.functional.normalize(emb, p=2, dim=1)
            embs.append(emb.cpu().numpy().astype(np.float32))
            if (i // batch_size) % 20 == 0:
                print(f'  {i+len(batch)}/{len(texts)} done')
    del mdl; torch.mps.empty_cache() if DEVICE.type == 'mps' else None
    return np.vstack(embs)

def load_or_compute(name, texts, model_id, pool, max_len=512, batch_size=8):
    cache = EMB_DIR / f'{name}.npz'
    if cache.exists():
        d = np.load(cache, allow_pickle=True)
        if d.get('hash', np.array(''))[()].decode() == hashlib.md5(json.dumps(texts[:20], ensure_ascii=False).encode()).hexdigest()[:8]:
            print(f'  [{name}] 캐시 로드: shape={d["emb"].shape}')
            return d['emb']
    print(f'  [{name}] 임베딩 계산 중 ({len(texts):,}개, model={model_id}, pool={pool})...')
    t0 = time.time()
    emb = encode_texts(texts, model_id, pool=pool, max_len=max_len, batch_size=batch_size)
    h = hashlib.md5(json.dumps(texts[:20], ensure_ascii=False).encode()).hexdigest()[:8]
    np.savez(cache, emb=emb, hash=np.array(h.encode()))
    print(f'  [{name}] 완료: {time.time()-t0:.1f}s, shape={emb.shape}')
    return emb

In [ ]:
# ---- 두 백본 임베딩 사전계산 (캐시 활용) ----
user_texts_list = [user_texts[u] for u in all_uids]
jd_texts_list   = [jd_texts[int(j)] for j in all_jids]

print('=== Qwen3-Embedding-0.6B (last_token pool) ===')
user_emb_qwen = load_or_compute('user_qwen3', user_texts_list, 'Qwen/Qwen3-Embedding-0.6B', pool='last', max_len=512, batch_size=4)
jd_emb_qwen   = load_or_compute('jd_qwen3',   jd_texts_list,   'Qwen/Qwen3-Embedding-0.6B', pool='last', max_len=512, batch_size=4)

print('\n=== KURE-v1 (mean pool, BGE-M3 base) ===')
user_emb_kure = load_or_compute('user_kure', user_texts_list, 'nlpai-lab/KURE-v1', pool='mean', max_len=512, batch_size=8)
jd_emb_kure   = load_or_compute('jd_kure',   jd_texts_list,   'nlpai-lab/KURE-v1', pool='mean', max_len=512, batch_size=8)

print(f'\n임베딩 shape — Qwen3 user: {user_emb_qwen.shape}, jd: {jd_emb_qwen.shape}')
print(f'임베딩 shape — KURE  user: {user_emb_kure.shape}, jd: {jd_emb_kure.shape}')

---
## Train/Held-out 분할 + 학습 데이터 생성 (exp-018과 동일 split)

In [ ]:
rng2 = np.random.default_rng(7)
shuf = list(all_uids); rng2.shuffle(shuf)
train_uids, held_uids = shuf[:800], shuf[800:]
train_idx = np.array([all_uids.index(u) for u in train_uids])
held_idx  = np.array([all_uids.index(u) for u in held_uids])
print(f'train users: {len(train_uids)}, held-out: {len(held_uids)}')

# 후보 sampling (exp-018과 동일: positive 8, hard-neg 6, easy-neg 8 per user)
rng3 = np.random.default_rng(42)
jid_role = {int(j): JP[int(j)]['role'] for j in all_jids}
jid_ind  = {int(j): JP[int(j)]['industry'] for j in all_jids}

def sample_candidates(uid, n_pos=8, n_hard=6, n_easy=8):
    u = UP[uid]
    pos  = [j for j in all_jids if jid_role[int(j)] in u['jobs']]
    hard = [j for j in all_jids if jid_role[int(j)] not in u['jobs'] and jid_ind[int(j)] in u['industries']]
    easy = [j for j in all_jids if jid_role[int(j)] not in u['jobs'] and jid_ind[int(j)] not in u['industries']]
    out = []
    for pool, n in [(pos, n_pos), (hard, n_hard), (easy, n_easy)]:
        if pool: out += list(rng3.choice(pool, size=min(n, len(pool)), replace=False))
    return [int(j) for j in out]

train_pairs = []
for uid in train_uids:
    for jid in sample_candidates(uid):
        train_pairs.append((uid, jid))
print(f'train_pairs: {len(train_pairs):,} (user × candidates)')

---
## 학습 head 정의 + Sweep 학습 + NDCG 평가

In [ ]:
import torch.nn as nn
import torch.nn.functional as F

class FusionHead(nn.Module):
    """입력: [user_emb ⊕ jd_emb ⊕ 5컴포넌트] → 매칭 점수."""
    def __init__(self, user_dim, jd_dim, comp_dim=5, arch='linear', hidden=128):
        super().__init__()
        in_dim = user_dim + jd_dim + comp_dim
        if arch == 'linear':
            self.net = nn.Linear(in_dim, 1)
        elif arch == 'mlp':
            self.net = nn.Sequential(
                nn.Linear(in_dim, hidden), nn.ReLU(), nn.Dropout(0.1),
                nn.Linear(hidden, 1)
            )
        else: raise ValueError(arch)
        self.in_dim = in_dim
    def forward(self, user_emb, jd_emb, comp):
        x = torch.cat([user_emb, jd_emb, comp], dim=-1)
        return self.net(x).squeeze(-1)

def ndcg_k(scores, labels, k=10):
    order = np.argsort(-scores, kind='stable')[:k]
    disc  = 1 / np.log2(np.arange(2, k + 2))
    dcg   = (labels[order] * disc[:len(order)]).sum()
    ideal = np.sort(labels)[::-1][:k]
    idcg  = (ideal * disc[:len(ideal)]).sum()
    return dcg / idcg if idcg > 0 else np.nan

def evaluate_full(head, user_emb_t, jd_emb_t, comp_t, label_mat, user_idx_subset):
    """held-out user × ALL 3k JD, 관점별 NDCG@10 평균."""
    head.eval()
    nd_per_persp = {p: [] for p in PERSPS}
    with torch.no_grad():
        for i in user_idx_subset:
            u_emb = user_emb_t[i:i+1].expand(N_JDS, -1)        # (3000, ud)
            scores = head(u_emb, jd_emb_t, comp_t[i]).cpu().numpy()  # (3000,)
            for p in PERSPS:
                L = label_mat[p][i]
                if L.max() == 0: continue
                nd_per_persp[p].append(ndcg_k(scores, L))
    return {p: float(np.mean(v)) if v else float('nan') for p, v in nd_per_persp.items()}

print('FusionHead 정의 완료')

In [ ]:
def train_head(user_emb_np, jd_emb_np, comp_mat, label_mat, train_pairs,
               persp='E', arch='linear', loss='mse', lr=1e-3, wd=1e-4,
               epochs=20, batch=512, verbose=False):
    """하나의 설정으로 head 학습."""
    torch.manual_seed(SEED)
    if torch.backends.mps.is_available(): torch.mps.manual_seed(SEED)
    
    ud_dim, jd_dim = user_emb_np.shape[1], jd_emb_np.shape[1]
    head = FusionHead(ud_dim, jd_dim, 5, arch=arch).to(DEVICE)
    opt = torch.optim.Adam(head.parameters(), lr=lr, weight_decay=wd)
    
    user_t = torch.tensor(user_emb_np, dtype=torch.float32, device=DEVICE)
    jd_t   = torch.tensor(jd_emb_np,   dtype=torch.float32, device=DEVICE)
    comp_t = torch.tensor(comp_mat,    dtype=torch.float32, device=DEVICE)
    
    # 학습 데이터 준비
    Ls = label_mat[persp]  # (999, 3000) int8
    train_user_idx = np.array([all_uids.index(uid) for uid, _ in train_pairs])
    train_jd_idx   = np.array([list(all_jids).index(jid) for _, jid in train_pairs])
    train_labels   = np.array([Ls[ui, ji] for ui, ji in zip(train_user_idx, train_jd_idx)], dtype=np.float32)
    
    n = len(train_pairs); perm = np.arange(n)
    losses = []
    for ep in range(epochs):
        np.random.shuffle(perm)
        epoch_loss = 0; nb = 0
        for bi in range(0, n, batch):
            idx = perm[bi:bi+batch]
            ui = train_user_idx[idx]; ji = train_jd_idx[idx]; lb = train_labels[idx]
            u_emb = user_t[ui]; j_emb = jd_t[ji]; c_in = comp_t[ui, ji]
            y = torch.tensor(lb, dtype=torch.float32, device=DEVICE)
            pred = head(u_emb, j_emb, c_in)
            if loss == 'mse':
                L = F.mse_loss(pred, y)
            elif loss == 'rank':
                # margin pair within batch
                diff_y = y.unsqueeze(0) - y.unsqueeze(1)
                diff_p = pred.unsqueeze(0) - pred.unsqueeze(1)
                mask = diff_y > 0
                if mask.sum() > 0:
                    L = F.relu(1.0 - diff_p[mask]).mean()
                else:
                    L = F.mse_loss(pred, y)
            opt.zero_grad(); L.backward(); opt.step()
            epoch_loss += L.item(); nb += 1
        losses.append(epoch_loss/nb)
        if verbose and (ep % 5 == 0 or ep == epochs-1):
            print(f'  ep{ep}: loss={losses[-1]:.4f}')
    return head, losses

print('train_head 정의 완료')

In [ ]:
# ---- Sweep: 백본 × 아키텍처 × 손실 × wd ----
configs = []
for backbone in ['qwen3', 'kure']:
    for arch in ['linear', 'mlp']:
        for loss in ['mse', 'rank']:
            for wd in [0.0, 1e-3]:
                configs.append({'backbone': backbone, 'arch': arch, 'loss': loss, 'wd': wd})
print(f'총 {len(configs)}개 설정 sweep')

results = []
for ci, cfg in enumerate(configs):
    print(f'\n[{ci+1}/{len(configs)}] {cfg}')
    user_e = user_emb_qwen if cfg['backbone'] == 'qwen3' else user_emb_kure
    jd_e   = jd_emb_qwen   if cfg['backbone'] == 'qwen3' else jd_emb_kure
    t0 = time.time()
    head, losses = train_head(user_e, jd_e, COMP_MAT, LABEL_MAT, train_pairs,
                              persp='E', arch=cfg['arch'], loss=cfg['loss'],
                              lr=1e-3, wd=cfg['wd'], epochs=15)
    train_time = time.time() - t0
    # 평가 (held-out 199명 × 3k JD)
    user_t = torch.tensor(user_e, dtype=torch.float32, device=DEVICE)
    jd_t   = torch.tensor(jd_e,   dtype=torch.float32, device=DEVICE)
    comp_t = torch.tensor(COMP_MAT, dtype=torch.float32, device=DEVICE)
    nd = evaluate_full(head, user_t, jd_t, comp_t, LABEL_MAT, held_idx)
    nd_mean = float(np.mean(list(nd.values())))
    print(f'  NDCG@10: {nd} → MEAN={nd_mean:.4f}, 학습 {train_time:.1f}s')
    results.append({**cfg, **{f'ndcg_{p}': round(nd[p], 4) for p in PERSPS},
                    'ndcg_mean': round(nd_mean, 4), 'final_loss': round(losses[-1], 4),
                    'train_time_s': round(train_time, 1)})
    del head; torch.mps.empty_cache() if DEVICE.type == 'mps' else None

res_df = pd.DataFrame(results).sort_values('ndcg_mean', ascending=False)
res_df.to_csv(OUT_DIR / 'sweep_results.csv', index=False)
print('\n=== Sweep 결과 (상위 5) ===')
print(res_df.head().to_string(index=False))

In [ ]:
# ---- Best config로 5관점 분기 head 학습 (관점별 독립) ----
best = res_df.iloc[0].to_dict()
print(f'★ Best config: {best}')
user_e = user_emb_qwen if best['backbone'] == 'qwen3' else user_emb_kure
jd_e   = jd_emb_qwen   if best['backbone'] == 'qwen3' else jd_emb_kure
user_t = torch.tensor(user_e, dtype=torch.float32, device=DEVICE)
jd_t   = torch.tensor(jd_e,   dtype=torch.float32, device=DEVICE)
comp_t = torch.tensor(COMP_MAT, dtype=torch.float32, device=DEVICE)

branched_heads = {}
branched_nd = {}
print('\n관점별 분기 head 학습...')
for p in PERSPS:
    t0 = time.time()
    head, losses = train_head(user_e, jd_e, COMP_MAT, LABEL_MAT, train_pairs,
                              persp=p, arch=best['arch'], loss=best['loss'],
                              lr=1e-3, wd=best['wd'], epochs=15)
    nd = evaluate_full(head, user_t, jd_t, comp_t, LABEL_MAT, held_idx)
    branched_heads[p] = head
    branched_nd[p] = nd[p]  # 관점 p로 학습 → 관점 p 평가
    print(f'  {p}: NDCG={nd[p]:.4f}  ({time.time()-t0:.1f}s)')

branched_mean = float(np.mean(list(branched_nd.values())))
print(f'\n관점별 분기 head 평균 NDCG@10: {branched_mean:.4f}')
print(f'통합 head(관점 E 학습 → 5관점 평가) 평균: {best["ndcg_mean"]:.4f}')

In [ ]:
# ---- V1(exp-018) vs V2 비교 + 독립 라벨 평가 ----
# V1: exp-018 golden weights
import pandas as pd
v1_golden = pd.read_csv(EXP / 'exp-018-learnable-fusion-head' / 'golden_weights.csv', index_col=0)
v1_arb    = pd.read_csv(EXP / 'exp-018-learnable-fusion-head' / 'arbitrary_weights.csv', index_col=0)
V1_GOLD = {p: v1_golden.loc[p].values.astype(np.float32) for p in PERSPS}
V1_ARB  = {p: v1_arb.loc[p].values.astype(np.float32) for p in PERSPS}

v1_gold_nd, v1_arb_nd = {p: [] for p in PERSPS}, {p: [] for p in PERSPS}
for p in PERSPS:
    for i in held_idx:
        L = LABEL_MAT[p][i]
        if L.max() == 0: continue
        v1_gold_nd[p].append(ndcg_k(COMP_MAT[i] @ V1_GOLD[p], L))
        v1_arb_nd[p].append(ndcg_k(COMP_MAT[i] @ V1_ARB[p], L))

comparison = {
    'V1 arbitrary (rule)':  {p: round(float(np.mean(v1_arb_nd[p])), 4) for p in PERSPS},
    'V1 golden (rule)':     {p: round(float(np.mean(v1_gold_nd[p])), 4) for p in PERSPS},
    'V2 unified (best cfg)': {p: best[f'ndcg_{p}'] for p in PERSPS},
    'V2 branched (perspective)': {p: round(branched_nd[p], 4) for p in PERSPS},
}
cmp_df = pd.DataFrame(comparison).T
cmp_df['MEAN'] = cmp_df.mean(axis=1).round(4)
print('=== V1 vs V2 (held-out 199 × 3k JD, 순환 라벨) ===')
print(cmp_df.to_string())
cmp_df.to_csv(OUT_DIR / 'v1_vs_v2_comparison.csv')

In [ ]:
# ---- 독립 라벨 평가 (exp-019/020 60+120쌍에 V2 점수 계산) ----
indep_csv = EXP / 'exp-020-independent-eval-stats' / 'eval_pairs.csv'
if indep_csv.exists():
    indep = pd.read_csv(indep_csv)
    if 'llm_label' in indep.columns:
        lbl_col = 'llm_label'
    elif 'label' in indep.columns:
        lbl_col = 'label'
    else:
        lbl_col = [c for c in indep.columns if 'label' in c.lower()][0]
    print(f'독립 라벨 컬럼: {lbl_col}, 쌍 수: {len(indep)}')
    
    # 각 (uid, jid) pair에 대해 V2 점수 계산 (관점 E 통합 head 사용)
    head_E = branched_heads['E']
    head_E.eval()
    v2_scores = []
    with torch.no_grad():
        for _, r in indep.iterrows():
            uid = r['userId'] if 'userId' in r else r.get('user_id')
            jid = int(r['job_id'])
            if uid not in all_uids or jid not in jd_texts:
                v2_scores.append(np.nan); continue
            ui = all_uids.index(uid); ji = list(all_jids).index(jid)
            ue = user_t[ui:ui+1]; je = jd_t[ji:ji+1]; ce = comp_t[ui:ui+1, ji]
            v2_scores.append(float(head_E(ue, je, ce).cpu().item()))
    indep['v2_score'] = v2_scores
    valid = indep.dropna(subset=['v2_score'])
    if len(valid) > 0:
        # per-user NDCG
        nds_v2 = []
        for uid, g in valid.groupby([c for c in ['userId','user_id'] if c in valid.columns][0]):
            if g[lbl_col].max() == 0: continue
            nds_v2.append(ndcg_k(g['v2_score'].values, g[lbl_col].values.astype(float)))
        v2_indep_ndcg = float(np.mean(nds_v2))
        print(f'\n★ V2 독립 라벨 NDCG@10: {v2_indep_ndcg:.4f} (N={len(nds_v2)} users)')
        print(f'   exp-020 V1 독립 라벨 NDCG@10 (gold_E): 0.9204 (참고)')
    else:
        print('독립 라벨 pair가 user/jd 집합 안에 없음 — skip')
        v2_indep_ndcg = float('nan')
    indep.to_csv(OUT_DIR / 'independent_label_v2_scores.csv', index=False)
else:
    print('eval_pairs.csv 없음')
    v2_indep_ndcg = float('nan')

In [ ]:
# ---- 메타 + 재현성 정보 저장 ----
meta = {
    'experiment': 'exp-023-v2-embedding-fusion-head', 'date': '2026-06-05',
    'env': ENV,
    'data': {'users': N_USERS, 'jds': N_JDS, 'train_users': len(train_uids), 'held_users': len(held_uids),
             'train_pairs': len(train_pairs)},
    'cache_hashes': {'user_text': user_text_hash, 'jd_text': jd_text_hash},
    'sweep_n_configs': len(configs),
    'best_config': {k: best[k] for k in ['backbone','arch','loss','wd','ndcg_mean']},
    'v1_vs_v2_held_out': comparison,
    'branched_per_perspective': {p: round(branched_nd[p], 4) for p in PERSPS},
    'v2_independent_ndcg': v2_indep_ndcg,
}
(OUT_DIR / 'meta.json').write_text(json.dumps(meta, indent=2, ensure_ascii=False, default=str))
print('저장 완료:', sorted(p.name for p in OUT_DIR.iterdir()))
print('\n핵심 결과:')
print(f'  V1 golden    (rule):   MEAN NDCG@10 = {cmp_df.loc["V1 golden (rule)", "MEAN"]:.4f}')
print(f'  V2 unified   ({best["backbone"]}/{best["arch"]}/{best["loss"]}): MEAN NDCG@10 = {cmp_df.loc["V2 unified (best cfg)", "MEAN"]:.4f}')
print(f'  V2 branched  (관점별 분기):  MEAN NDCG@10 = {cmp_df.loc["V2 branched (perspective)", "MEAN"]:.4f}')
print(f'  V2 독립라벨 NDCG@10:  {v2_indep_ndcg:.4f}  (참고: exp-020 V1=0.9204)')